# PhenoPY × Google Earth Engine — Torres del Paine (Patagonia)

This notebook pulls **real, cloud-contaminated** optical NDVI **with a
per-observation quality band** from three sensors over Torres del Paine
(~51° S, Chilean Patagonia — one of the cloudiest places on Earth, so the
growing season is hard to see without quality weighting).

**Design (important).** Earth Engine is used here only as an **example data
source — it is *not* a dependency of PhenoPY.** The split is deliberate:

- **Getting data into xarray** (the `gee_to_xarray` helper below) is example
  code; swap it for local rasters / NetCDF and everything downstream is identical.
- **Decoding QA into weights** is done by the library, `phenopy.qa.qa_to_weight`,
  which is **source- and sensor-agnostic** (works on any xarray QA band).
- **The phenology algorithms** only ever see a generic `weight` array in [0, 1];
  they never know which sensor it came from.

> ⚠️ **You must run this yourself.** Earth Engine auth is an interactive Google
> login and needs a **Google Cloud project**; it cannot run in CI. The cells are
> saved *unexecuted*. Requires the `[gee]` extra (`earthengine-api`, `xee`).

## 1. One-time setup: Earth Engine account + Google Cloud project

1. **Register** your Google account at <https://code.earthengine.google.com> and
   accept the terms (once).
2. **Create a Cloud project**: the Code Editor prompts you to, or go to
   <https://console.cloud.google.com> → *New Project* and note its **project ID**
   (e.g. `ee-yourname`).
3. **Authenticate** once per machine: easiest is `earthengine authenticate` in a
   terminal (it captures the token automatically — nothing to paste). Or run
   `ee.Authenticate()` below.
4. Put your **project ID** in `PROJECT` and run `ee.Initialize(...)`.

In [ ]:
import ee

PROJECT = "ee-your-project-id"   # <-- EDIT: your Google Cloud project ID

# Run once per machine; comment out after the token is saved.
ee.Authenticate()

# xee works best against the high-volume endpoint.
ee.Initialize(project=PROJECT, opt_url="https://earthengine-highvolume.googleapis.com")
print("Earth Engine initialised for project:", PROJECT)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import phenopy
import phenopy.qa as pqa                       # <-- the source-agnostic QA decoder
from phenopy.plotting import PhenoPlot, display_map
print("phenopy", phenopy.__version__, "| xarray", xr.__version__)

## 2. Area of interest, period & common grid

A small box over the eastern Patagonian steppe of Torres del Paine (vegetated;
avoids the rock/ice/lakes of the massif). `xee 0.1.1` samples onto an explicit
grid (CRS + affine transform + width/height); using **one common grid for all
three sensors** makes them directly comparable, pixel for pixel.

Southern Hemisphere: the season peaks around the New Year (Dec–Feb), so it is
split on a calendar-DOY axis — a natural fit for PhenoPY's `southern=True`.

In [ ]:
# [west, south, east, north] around the Laguna Amarga / Sarmiento steppe
WEST, SOUTH, EAST, NORTH = -72.95, -51.01, -72.89, -50.97
geom = ee.Geometry.Rectangle([WEST, SOUTH, EAST, NORTH])

START, END = "2019-01-01", "2024-01-01"   # 5 austral seasons

CRS = "EPSG:4326"
PIXEL = 0.0025                                          # ~250 m in latitude degrees
WIDTH, HEIGHT = round((EAST - WEST) / PIXEL), round((NORTH - SOUTH) / PIXEL)
CRS_TRANSFORM = (PIXEL, 0.0, WEST, 0.0, -PIXEL, NORTH)  # affine: (sx, 0, x0, 0, -sy, y0)
SHAPE_2D = (WIDTH, HEIGHT)
print(f"common grid: {WIDTH} x {HEIGHT} px")

# quick interactive check of the box (folium; needs the [plot] extra)
display_map((WEST, EAST), (SOUTH, NORTH))

## 3. Per-sensor collections: NDVI + the *raw* QA band

Each helper returns an `ee.ImageCollection` with `ndvi` plus that sensor's **raw
quality band** — note we do **not** decode QA here; that is the library's job
(§4). Computing NDVI from surface-reflectance bands and the GEE specifics all
live in this example layer, not in PhenoPY.

- **MODIS MOD13Q1** — precomputed NDVI + `SummaryQA` (ordinal 0–3).
- **Landsat 8/9 C2 L2** — NDVI from scaled SR bands + `QA_PIXEL` (bitmask).
- **Sentinel-2 SR Harmonized** — NDVI from B8/B4 + `SCL` (scene classes).

In [ ]:
def modis(geom, start, end):
    ic = ee.ImageCollection("MODIS/061/MOD13Q1").filterBounds(geom).filterDate(start, end)
    def prep(img):
        ndvi = img.select("NDVI").multiply(0.0001).rename("ndvi")
        return ndvi.addBands(img.select("SummaryQA")).set(
            "system:time_start", img.get("system:time_start"))
    return ic.map(prep).select(["ndvi", "SummaryQA"])


def landsat(geom, start, end):
    def prep(img):
        sr = img.select(["SR_B4", "SR_B5"]).multiply(0.0000275).add(-0.2)   # C2 SR scaling
        ndvi = sr.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")    # (NIR-Red)/(NIR+Red)
        return ndvi.addBands(img.select("QA_PIXEL")).set(
            "system:time_start", img.get("system:time_start"))
    l8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filterBounds(geom).filterDate(start, end)
    l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(geom).filterDate(start, end)
    return l8.merge(l9).map(prep).select(["ndvi", "QA_PIXEL"])


def s2(geom, start, end):
    ic = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(geom).filterDate(start, end)
    def prep(img):
        ndvi = img.normalizedDifference(["B8", "B4"]).rename("ndvi")
        return ndvi.addBands(img.select("SCL")).set(
            "system:time_start", img.get("system:time_start"))
    return ic.map(prep).select(["ndvi", "SCL"])


# sensor -> (collection builder, phenopy.qa registry key)
SENSORS = {"MODIS": (modis, "MOD13Q1"), "Landsat": (landsat, "LANDSAT_C2"), "S2": (s2, "S2_SCL")}

### The library decoder — `phenopy.qa`

`phenopy.qa.qa_to_weight(qa, spec)` turns any xarray QA band into weights in
[0, 1]. `spec` can be a **built-in key**, a **declarative dict**, or a **callable**
for anything exotic — so a new sensor is one line, not a new code path. The four
patterns it covers: `remap` (ordinal), `good_classes` (labels), `bad_bits`
(bitmask), `max_value`/`min_value` (threshold).

In [ ]:
print("built-in specs:", pqa.list_qa_specs())
print("S2_SCL rule    :", pqa.get_qa_spec("S2_SCL"))

# A less-typical sensor needs no library change — just pass a dict ...
cloudprob_spec = {"max_value": 30}                      # keep cloud-probability <= 30 %
# ... or a callable for anything that fits no pattern:
custom = lambda qa: (qa < 2).astype(float)              # noqa: E731  (illustrative)
print("ad-hoc dict / callable are accepted directly by qa_to_weight()")

## 4. Pull each cube (`gee_to_xarray`) and decode QA with `phenopy.qa`

`gee_to_xarray` is the **example** GEE → xarray bridge (swap it for any loader).
For each sensor we then call the **library** `qa_to_weight` on the downloaded raw
QA band. (S2 is the densest collection, but on this tiny grid all three take only
seconds.)

In [ ]:
def gee_to_xarray(collection, crs, crs_transform, shape_2d):
    """Example GEE -> xarray bridge. Earth Engine is NOT a PhenoPY dependency;
    this helper lives in the example, not the library. Swap it for rioxarray /
    open_dataset on local files and everything below is unchanged."""
    ds = xr.open_dataset(
        collection, engine="ee", crs=crs, crs_transform=crs_transform, shape_2d=shape_2d
    ).load()
    ds = ds.sortby("time")
    return ds.assign_coords(
        doy=("time", ds["time"].dt.dayofyear.data),
        year=("time", ds["time"].dt.year.data),
    )


cubes = {}
for name, (build, qa_key) in SENSORS.items():
    ds = gee_to_xarray(build(geom, START, END), CRS, CRS_TRANSFORM, SHAPE_2D)
    qa_band = pqa.get_qa_spec(qa_key)["band"]            # which band holds the QA
    weight = pqa.qa_to_weight(ds[qa_band], qa_key)       # <-- library, sensor-agnostic
    cubes[name] = (ds["ndvi"], weight)
    print(f"{name:8s} -> {dict(ds['ndvi'].sizes)} "
          f"({int(np.isfinite(ds['ndvi']).sum())} finite NDVI, mean QA weight {float(weight.mean()):.2f})")

## 5. The problem QA weighting solves

Colour each observation of one pixel by its decoded QA weight: in cloudy Patagonia
the **low-weight (red) points sit well below** the clean (green) ones — the
asymmetric, downward contamination that drags an unweighted smoother down.

In [ ]:
name = "S2"               # the densest / cloudiest series shows it best
ndvi, weight = cubes[name]
yc, xc = ndvi.sizes["y"] // 2, ndvi.sizes["x"] // 2
ts, wt = ndvi.isel(y=yc, x=xc), weight.isel(y=yc, x=xc)

plt.figure(figsize=(9, 4))
sc = plt.scatter(ts["doy"], ts, c=wt, cmap="RdYlGn", vmin=0, vmax=1, s=14)
plt.colorbar(sc, label="QA weight")
plt.xlabel("Day of year"); plt.ylabel("NDVI")
plt.title(f"{name}: observations coloured by phenopy.qa weight (low = cloud/shadow/snow)")
plt.show()

## 6. Weighted vs unweighted reconstruction

Three reconstructions of the same cloudy pixel:
- **unweighted** Whittaker — dragged down by the cloud-contaminated lows;
- **QA-weighted** Whittaker — `weights` from `phenopy.qa` down-weight bad
  observations (`PhenoShape(weights=...)`);
- **upper-envelope** (Chen / TIMESAT wTSM) — tracks the noise-free upper envelope,
  no QA band needed.

In cloudy Patagonia the unweighted curve sits far too low; both weighted methods
recover the real vegetation signal (most dramatic for Sentinel-2 / Landsat, whose
raw per-scene data is heavily cloud-contaminated).

In [ ]:
name = "S2"                     # the cloudiest series -> clearest contrast
ndvi, weight = cubes[name]
yc, xc = ndvi.sizes["y"] // 2, ndvi.sizes["x"] // 2
rp = {"lmbd": 50}

unweighted = ndvi.pheno.PhenoShape(interpolType="whittaker", recon_params=rp, rollWindow=None)
qa_weighted = ndvi.pheno.PhenoShape(
    interpolType="whittaker", recon_params=rp, rollWindow=None, weights=weight
)
envelope = ndvi.pheno.PhenoShape(
    interpolType="upper_envelope",
    recon_params={"base": "whittaker", "n_iter": 5, "base_params": rp},
    rollWindow=None,
)

plt.figure(figsize=(9, 5))
ts, wt = ndvi.isel(y=yc, x=xc), weight.isel(y=yc, x=xc)
sc = plt.scatter(ts["doy"], ts, c=wt, cmap="RdYlGn", vmin=0, vmax=1, s=12)
plt.colorbar(sc, label="QA weight")
plt.plot(unweighted["doy"], unweighted.isel(y=yc, x=xc), color="0.4", ls="--", lw=2, label="unweighted")
plt.plot(qa_weighted["doy"], qa_weighted.isel(y=yc, x=xc), color="C0", lw=2.5, label="QA-weighted Whittaker")
plt.plot(envelope["doy"], envelope.isel(y=yc, x=xc), color="C3", lw=2, label="upper-envelope (wTSM/Chen)")
plt.xlabel("Day of year"); plt.ylabel("NDVI"); plt.legend(loc="upper left")
plt.title(f"{name}: weighting / upper-envelope recover the cloud-suppressed signal")
plt.show()

## 7. Cache locally — the real-data fixture

Save each `(ndvi, weight)` cube under `examples/data/` so the regression tests and
further analysis don't hit Earth Engine again. (`examples/data/` is git-ignored;
downsample to a tiny committable fixture later if useful.)

In [ ]:
from pathlib import Path

# robust whether the notebook's CWD is the repo root or the examples/ folder
here = Path.cwd()
outdir = here / "examples" / "data" if (here / "examples").is_dir() else here / "data"
outdir.mkdir(parents=True, exist_ok=True)
for name, (ndvi, weight) in cubes.items():
    out = outdir / f"torres_del_paine_{name}.nc"
    xr.Dataset({"ndvi": ndvi, "weight": weight}).to_netcdf(out)
    print("wrote", out)

## 8. Recap

The weights from `phenopy.qa` feed the library's **weighted reconstruction**
(now implemented):

- `PhenoShape(weights=...)` — weighted Whittaker driven by the QA weights;
- `interpolType="upper_envelope"` — Chen / TIMESAT wTSM upper-envelope iteration,
  no QA band needed;
- the default unweighted path is unchanged (byte-identical, golden tests green).

From here the cached `examples/data/` cubes are a real-data fixture for regression
tests and for comparing the LSP metrics (SOS/POS/EOS …) extracted from weighted
vs unweighted curves.